# RAGAS

## Imports

In [120]:
import os
import numpy as np
import pandas as pd
import json
import nltk
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
import re
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

from openai import AzureOpenAI
from openai import AsyncAzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from datasets import Dataset

from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
from ragas.llms import llm_factory
import nest_asyncio
from ragas import aevaluate 
from ragas import RunConfig

# only first time
# nltk.download('stopwords')

C:\Users\verkad004\AppData\Local\Temp\ipykernel_17468\3051464301.py:18: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_17468\3051464301.py:18: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_17468\3051464301.py:18: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' inst

## Dataset loading

In [110]:
# path for the jurisprudence corpus
jurisprudence_path = 'jurisprudence_omgevingswet.jsonl'

# load JSONL into DataFrame
jur_data = []
with open(jurisprudence_path, 'r', encoding='utf-8') as f:
    for line in f:
        jur_data.append(json.loads(line))

df_jurisprudence = pd.DataFrame(jur_data)
print(f"Jurisprudence dataset shape: {df_jurisprudence.shape}")
print(f"Columns: {df_jurisprudence.columns.tolist()}")


# path for the legislation corpus
legislation_path = 'legislation_omgevingswet.jsonl'

# load JSONL into DataFrame
leg_data = []
with open(legislation_path, 'r', encoding='utf-8') as f:
    for line in f:
        leg_data.append(json.loads(line))

df_legislation = pd.DataFrame(leg_data)
print(f"Legislation dataset shape: {df_legislation.shape}")
print(f"Columns: {df_legislation.columns.tolist()}")

#corpus generation
jurisprudence_docs = [
    Document(
        page_content=row['full_text'], 
        metadata={
            "ecli": row['ecli'], 
            "title": row['title'], 
            "type": "jurisprudence",
            "updated": row['updated']
        }
    )
    for _, row in df_jurisprudence.iterrows()
]

legislation_docs = [
    Document(
        page_content=row['text'], 
        metadata={
            "id": row['id'], 
            "title": row['title'], 
            "source": row['source_type'],
            "type": "legislation"
        }
    )
    for _, row in df_legislation.iterrows()
]

dutch_stopwords = set(stopwords.words('dutch'))
stemmer = SnowballStemmer("dutch")

def preprocess_func(text):
    # 1. Lowercasing
    text = text.lower()
    # 2. Tokenization (alleen woorden)
    tokens = re.findall(r'\w+', text)
    # 3. Stopword removal & 4. Stemming
    return [stemmer.stem(t) for t in tokens if t not in dutch_stopwords]


Jurisprudence dataset shape: (3906, 5)
Columns: ['ecli', 'title', 'updated', 'summary', 'full_text']
Legislation dataset shape: (725, 4)
Columns: ['source_type', 'id', 'title', 'text']


## RAGAs Dataset loading

### Algemene dataset met questions en ground truths

In [111]:
json_path = os.path.join('..', 'data', 'QA_pairs_evaluation.json')

with open(json_path, 'r') as f:
    eval_dataset_json = json.load(f)

### Context retrieval

In [112]:
# Definieer de parameters in een dictionary
leg_params = {"k1": 1.2, "b": 0.75}
jur_params = {"k1": 1.5, "b": 0.85}

# Geef ze mee tijdens de initialisatie
legislation_retriever = BM25Retriever.from_documents(
    legislation_docs, 
    preprocess_func=preprocess_func,
    bm25_params=leg_params
)
jurisprudence_retriever = BM25Retriever.from_documents(
    jurisprudence_docs, 
    preprocess_func=preprocess_func,
    bm25_params=jur_params
)

legislation_retriever.k = 3
jurisprudence_retriever.k = 3


In [113]:
print(f"Aantal documenten in wetgeving index: {len(legislation_retriever.docs)}")
print(f"Aantal documenten in jurisprudentie index: {len(jurisprudence_retriever.docs)}")

Aantal documenten in wetgeving index: 725
Aantal documenten in jurisprudentie index: 3906


### Answer generation

#### Azure OpenAI config

In [114]:
env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

if not endpoint or not api_version or not deployment:
    raise RuntimeError("Azure OpenAI env-variabelen ontbreken.")

# Initialiseer Azure AD token provider
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

# Initialiseer de Azure Client
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)


In [115]:
# 4. De gecombineerde loop
questions = []
all_contexts = []
ground_truths = []
all_answers = []

print("Start retrieval en generatie...")

for item in eval_dataset_json:
    q = item['question']
    gt = item['ground_truth']
    
    # --- RETRIEVAL ---
    legis_docs = legislation_retriever.invoke(q)
    juris_docs = jurisprudence_retriever.invoke(q)
    
    # Maak een lijst van strings voor RAGAS en een samengevoegde string voor de prompt
    ctx_list = [doc.page_content for doc in legis_docs + juris_docs]
    context_text = "\n\n".join(ctx_list)
    
    # --- GENERATIE ---
    resp = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": "Beantwoord de vraag uitsluitend op basis van de context."},
            {"role": "user", "content": f"Context: {context_text}\n\nVraag: {q}"},
        ],
        temperature=0 
    )
    answer = resp.choices[0].message.content
    
    # --- OPSLAAN IN LIJSTEN ---
    questions.append(q)
    all_contexts.append(ctx_list) # RAGAS verwacht een lijst van strings
    ground_truths.append(gt)
    all_answers.append(answer)

Start retrieval en generatie...


### RAGAs dataset generation

In [116]:
# 5. Dataset maken voor RAGAS
ragas_input_dict = {
    "question": questions,
    "contexts": all_contexts,
    "answer": all_answers,
    "ground_truth": ground_truths
}

ragas_eval_dataset = Dataset.from_dict(ragas_input_dict)

# 1. Opslaan als JSON (aanbevolen voor Ragas)
ragas_eval_dataset.to_json("test_ragas.jsonl")

print("Dataset is opgeslagen als test_ragas.jsonl")

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 50.08ba/s]

Dataset is opgeslagen als test_ragas.jsonl


## RAGAs scores

In [130]:
# Maak een configuratie die rustiger aan doet
async_client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)

evaluator_llm = llm_factory(
    model=deployment,
    client=async_client)

config = RunConfig(
    timeout=240,      # Wacht langer op Azure
    max_retries=20,  # Probeer vaker opnieuw bij een 429 error
    max_wait=60,      # Wacht maximaal een minuut tussen pogingen
    max_workers=2
)

# 1. Sta geneste async loops toe (cruciaal voor notebooks)
nest_asyncio.apply()

# 2. De metrics initialiseren (precies zoals we hadden, maar let op de imports)
# Gebruik 'FactualCorrectness' of 'AnswerCorrectness' afhankelijk van je exacte versie
metrics = [
    Faithfulness(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm), 
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]

# 3. Voer de evaluatie uit met 'await aevaluate'
# Dit omzeilt de TypeError die 'evaluate' geeft in v0.4.x
result = await aevaluate(
    dataset=ragas_eval_dataset,
    metrics=metrics,
    run_config=config
)

print(result)

C:\Users\verkad004\AppData\Local\Temp\ipykernel_17468\2464488374.py:40: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = await aevaluate(
Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]Exception raised in Job[0]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    The output is incomplete due to a max_tokens length limit.
</exception>
<completion>
    ChatCompletion(id='chatcmpl-DUY4sLMnPSgxoALXmCY1euSBvtNkE', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "statements": [\n        {\n            "statement": "Een omgevingsvergunning voor een dakterras op een gemeentelijk monument kan worden verleend, zelfs als het hekwerk de maximale bouwhoogte overschrijdt, mits aan bepaalde voorwaarden wordt voldaan.",\n            "reason"

{'faithfulness': 0.8450, 'factual_correctness(mode=f1)': 0.4067, 'context_precision': 0.5179, 'context_recall': 0.5389}


# Voorbeelden test data

In [ ]:
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
from openai import AsyncAzureOpenAI

async_client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)

evaluator_llm = llm_factory(
    model=deployment,
    client=async_client)

## Faithfullness

In [ ]:
# Create metric
scorer = Faithfulness(llm=evaluator_llm)

# Evaluate
result = await scorer.ascore(
    user_input="When was the first super bowl?",
    response="The first superbowl was held on Jan 15, 1967",
    retrieved_contexts=[
        "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
    ]
)
print(f"Faithfulness Score: {result.value}")

Faithfulness Score: 1.0


## Context precision

In [ ]:
from ragas.metrics.collections import ContextPrecision
scorer = ContextPrecision(evaluator_llm)

# Evaluate
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=[
        "The Eiffel Tower is located in Paris.",
        "The Brandenburg Gate is located in Berlin."
    ]
)
print(f"Context Precision Score: {result.value}")

Context Precision Score: 0.9999999999


## Context recall

In [ ]:
from ragas.metrics.collections import ContextRecall

# Create metric
scorer = ContextRecall(llm=evaluator_llm)

# Evaluate
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    retrieved_contexts=["Paris is the capital of France."],
    reference="The Eiffel Tower is located in Paris."
)
print(f"Context Recall Score: {result.value}")

Context Recall Score: 0.0


## Answer accuracy

In [ ]:
from ragas.metrics.collections import AnswerAccuracy

# Create metric
scorer = AnswerAccuracy(llm=evaluator_llm)

# Evaluate
result = await scorer.ascore(
    user_input="When was Einstein born?",
    response="Albert Einstein was born in 1879.",
    reference="Albert Einstein was born in 1879."
)
print(f"Answer Accuracy Score: {result.value}")

Answer Accuracy Score: 1.0


## Context relevance

In [ ]:
from ragas.metrics.collections import ContextRelevance

# Create metric
scorer = ContextRelevance(llm=evaluator_llm)

# Evaluate
result = await scorer.ascore(
    user_input="When and Where Albert Einstein was born?",
    retrieved_contexts=[
        "Albert Einstein was born March 14, 1879.",
        "Albert Einstein was born at Ulm, in Württemberg, Germany.",
    ]
)
print(f"Context Relevance Score: {result.value}")

Context Relevance Score: 1.0
